# Getting Started with ukpyn

Welcome to **ukpyn**! 

**ukpyn** is a Python package aimed at improving accessibility of the data hosted on [UK Power Networks' Open Data Portal](https://ukpowernetworks.opendatasoft.com/).
The Python package is created and maintained by UK Power Networks DSO Data Science & Development Team, you can contact us via the [ukpyn GitHub repository](https://github.com/UKPN-DSO/ukpyn) issues board to start a conversation about the package and get any help. 

We have prepared many tutorials and examples of how you might interact with the data on Open Data Portal, and combine them to unlock insights and value. We hope you find this information useful!

In this first tutorial, you will learn how to:

1. Install ukpyn
2. Set your API key
3. Create a client and list datasets
4. Read records from a dataset
5. Export data in common formats

This notebook is written for beginners, so each step explains what is happening and why.
We do assume that you have some knowledge of python with it installed and are able to work with a [Jupyter Notebook](https://jupyter.org/).

Let's begin!

## 1. Installation

Before you can use `ukpyn`, you need to install it in your Python environment.

### Why you need to install `ukpyn`?

`ukpyn` is a Python package (a reusable library) for working with UK Power Networks open data and is hosted on the Python Package Index (PyPI), a repository of software for the Python programming language.

Installing it adds that library to your Python environment so `import ukpyn` within notebooks and scripts will find the library.
Ensure you already have your Python environment setup already so that you are installing `ukpyn` there.

Without installation, Python does not know what `ukpyn` is as it won't find it in the environment and importing it will fail.

### What are `pip` and `uv`?

- `pip` is Python’s standard package installer.
- `uv` is a newer, faster package/dependency tool that can also install packages.

You can check you have these installed by simply typing `pip` or `uv` into your terminal and see if you get the appropriate response.
We recommend using `uv` for installations, found [here](https://docs.astral.sh/uv/getting-started/installation/).

### Installing the package

The `ukpyn` package has it basic installation and full installation options.
If you are just going to use the package on your own and not work through the tutorials/examples, then you only need the basic installation.
If you want to work with these tutorials, you will need all the additional dependencies.


---
_BASIC INSTALLATION_

Run these in a **terminal**, not inside a Python code cell.
In VS Code, open **Terminal → New Terminal** and run one of the following:

**Option A: using pip**
```bash
pip install ukpyn
```

**Option B: using uv (faster, recommended)**
```bash
uv add ukpyn
```

Both options install the `ukpyn` and its required dependencies (for example, `httpx` and `pydantic`).


---
_FULL INSTALLATION_

As we wish to be able to run all the tutorials and perhaps contribute to the development of `ukpyn`, we need the additional dependencies.

The Tutorials contain additional dependencies that are unassociated to the client manager itself. For example, we use some geospatial python packages in `tutorials/08-geospatial-data.ipynb`.
To operate within our tutorials, you will need to install additional dependencies by running one of the following:

**Option A: using pip**
```bash
pip install "ukpyn[all]"
```

**Option B: using uv (faster, recommended)**
```bash
uv add "ukpyn[all]"
```

## 2. Setting Up Your API Key
All of our data is available on the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/).
There you can manually export to recognisable file formats like `.csv` and `.xlsx`. 
However, it is much more efficient for code projects to have a computer program fetch the data for you. That is why all our datasets are accessible via API.

### What is an API?
API stands for **Application Programming Interface**. It is a set of rules that lets one software system request data or services from another, so your code can talk directly to platforms like the UKPN Open Data Portal automatically.

Where data is locked behind user accounts, as our Open Data Portal requires, then we need to give your code some credentials to access the data. We call this an API key.

The API key should be kept private and not hardcoded into your scripts. To do this, you have options available to you. Firstly, let's generate your API key.

### Generate your API key

1. Open the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/)
2. Sign in (or create an account)
3. Open your [profile settings](https://ukpowernetworks.opendatasoft.com/account/) from the top-right menu
4. Go to **Portal Settings → API Keys**
5. Select **Generate a new API key** and give it a name like `ukpyn`
6. Copy the generated key

### Save the key as an environment variable
`ukpyn` looks for your API key from your environmental variables looking for a variable named `UKPN_API_KEY`.
You will need to create this environment variable for `ukpyn` to find. There are a few options.

**Using a .env file (recommended):**
Create a new file called `.env` in the current working directory of the notebook session, this is normally the project root where you opened your IDE. 
We use `dotenv` that looks in the current directory then pareant directories for the .env file.
If you are still unsure where that is, you can print it from your terminal you can run the following:

```bash
python

import os; print(os.getcwd())
# a directory path will print to the screen, e.g. C:\projects\ukpyn

quit()
```
Now you have made your `.env` file, open it with software like Notepad or Notepad++ and add the following, replacing `your_api_key_here` with the key generated in the previous step:

```bash
UKPN_API_KEY=your_api_key_here
```

**Linux/macOS:**
```bash
export UKPN_API_KEY="your-api-key-here"
```

**Windows (Command Prompt):**
```cmd
set UKPN_API_KEY=your-api-key-here
```

**Windows (PowerShell):**
```powershell
$env:UKPN_API_KEY = "your-api-key-here"
```

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

# Verify your API key is set (don't print the actual key!)
if os.getenv("UKPN_API_KEY"):
    print("API key is configured!")
else:
    print("Warning: UKPN_API_KEY environment variable is not set.")
    print("Some API features may be limited without authentication.")

API key is configured!


## 3. Creating a Client and Listing Datasets

A **client** is a Python object that sends requests to the API and returns results.
In this notebook, `UKPNClient` uses **async** methods, so API calls use `await`.

**What does async mean here?**

- Async methods are useful for network calls, where Python waits for the server response without blocking everything else.
- `await` means: “pause this function here, let other tasks run, then resume when result is ready.”
- When using the `UKPNClient`, you will regularly see the command `await` before immediately it, which just tells your program to relax whilst the client speaks to the Open Data Portal.

Let's create the **client**:

In [2]:
from ukpyn import UKPNClient

# Create a client instance
# The API key is automatically loaded from UKPN_API_KEY environment variable
client = UKPNClient()

# Or, pass the API key directly (not recommended for production code)
# client = UKPNClient(api_key="your-api-key-here")

# Display a summary of the API client
print(client.summary())

UKPNClient(api_url='https://ukpowernetworks.opendatasoft.com/api/explore/v2.1', timeout=30, has_api_key=True, client_state='closed')


### Listing Available Datasets

By this point, you have `ukpyn` installed, your API key configured, and have a `client` ready to go! 
Now the fun begins.

Let's start by listing datasets so you can choose one to work with.
The example below requests the first 5 datasets and prints their summaries.

In [3]:
# List the first 10 datasets
response = await client.list_datasets(limit=10)

print(f"Total datasets available: {response.total_count}")
print("Dataset summaries:\n")

for item in response.datasets:
    dataset = item.dataset
    print("-" * 60)
    print(dataset.summary())
    print()

# Expected output:
# Dataset(id='...', title='...', has_records=..., records=..., fields=...)

Total datasets available: 135
Dataset summaries:

------------------------------------------------------------
Dataset(id='ukpn-shlaa', title='Strategic Housing Land Availability Assessment (SHLAA)', has_records=True, records=8516, fields=8)

------------------------------------------------------------
Dataset(id='ukpn-sensitivity-factors-export', title='Sensitivity Factors (Export)', has_records=True, records=562064, fields=5)

------------------------------------------------------------
Dataset(id='ltds-table-4b-earth-fault-level', title='Long Term Development Statement (LTDS) Table 4b Fault level data - Earth', has_records=True, records=2037, fields=13)

------------------------------------------------------------
Dataset(id='ltds-table-8-gt-95-perc-fault-data', title='Long Term Development Statement (LTDS) Table 8 >95% Fault Data', has_records=True, records=43, fields=8)

------------------------------------------------------------
Dataset(id='ltds-table-3b-load-data-true', title='

### Searching for Datasets

Use `search` when you only know a keyword (for example, `smart` or `flexibility`).
This helps you quickly find likely dataset IDs before fetching records.

In [4]:
# Search for datasets containing "smart" in their metadata
response = await client.list_datasets(search="smart", limit=5)

print(f"Found {response.total_count} datasets matching 'smart'\n")

for item in response.datasets:
    print(f"- {item.title} ({item.id})")

Found 135 datasets matching 'smart'

- Strategic Housing Land Availability Assessment (SHLAA) (ukpn-shlaa)
- Sensitivity Factors (Export) (ukpn-sensitivity-factors-export)
- Long Term Development Statement (LTDS) Table 4b Fault level data - Earth (ltds-table-4b-earth-fault-level)
- Long Term Development Statement (LTDS) Table 8 >95% Fault Data (ltds-table-8-gt-95-perc-fault-data)
- Long Term Development Statement (LTDS) Table 3b True Peak Demand (ltds-table-3b-load-data-true)


### Getting Dataset Details

Once you have a dataset ID, request full details.
This includes useful metadata and the list of available fields (columns).

In [5]:
# Get details for a specific dataset
# Replace with an actual dataset ID from the list above
dataset_id = "ukpn-smart-meter-installation-volumes"
dataset = await client.get_dataset(dataset_id)

# Detailed, human-readable breakdown (renders rich HTML in notebooks)
dataset.details()

Dataset: Smart Meter Installation Volumes
Dataset ID: ukpn-smart-meter-installation-volumes
Has records: True
Records: 121
URL: https://ukpowernetworks.opendatasoft.com/explore/dataset/ukpn-smart-meter-installation-volumes
Fields (total): 10
- smart (int), The number of smart meters
- not_smart (int), The number of analogue meters
- percentage_smart (double), % of total meters which are smart
- geo_shape (geo_shape), No description
- la_district_code (text), ladcd 
- la_district_name (text), ladnm
- geo_point (geo_point_2d), No description
- local_authority (text), The local authority
- local_authority_code (text), No description
- local_authority_geo_shape (geo_shape), No description

## 4. Fetching Records from a Dataset

After choosing a dataset, you can fetch rows (called **records**) from it.
Start with a small `limit` so you can inspect the structure safely.

In [6]:
# Fetch 5 records from the dataset
dataset_id = "ukpn-smart-meter-installation-volumes"
records_response = await client.get_records(dataset_id, limit=5)

print(f"Total records in dataset: {records_response.total_count}")
print(f"Records returned: {len(records_response.records)}")
print("\n" + "=" * 60)

# Display each record's fields in a readable format
for i, record in enumerate(records_response.records, 1):
    print(f"\nRecord {i} (ID: {record.id})")
    print("-" * 40)
    if record.fields:
        for key, value in record.fields.items():
            print(f"  {key}: {value}")

Total records in dataset: 121
Records returned: 5


Record 1 (ID: None)
----------------------------------------
  smart: 15290
  not_smart: 6309
  percentage_smart: 70.79031436640585
  geo_shape: {'type': 'Feature', 'geometry': {'coordinates': [[[[-0.522784347, 51.435300447], [-0.519015023, 51.434366242], [-0.512014259, 51.431227398], [-0.51040838, 51.42978358], [-0.509652553, 51.426220265], [-0.510041327, 51.423641662], [-0.512653169, 51.419571671], [-0.511560589, 51.416898833], [-0.506390048, 51.417311404], [-0.503067179, 51.416104579], [-0.501815429, 51.413880986], [-0.503806244, 51.411262634], [-0.500479505, 51.410802009], [-0.500843109, 51.413096968], [-0.499791413, 51.41407984], [-0.495530844, 51.413325652], [-0.493222601, 51.411887853], [-0.492536844, 51.410308982], [-0.492967845, 51.40537773], [-0.49116427, 51.402472509], [-0.491966108, 51.400470298], [-0.491101619, 51.398738986], [-0.487882285, 51.396756069], [-0.486578916, 51.394564896], [-0.486894816, 51.392087985], [-0.486

### Filtering Records

Use filters to return only the records you care about.
`ukpyn` supports ODSQL filters (OpenDataSoft Query Language) through query parameters like `where` and `order_by`.

In [10]:
# Example: Filter records with a WHERE clause
# Note: Adjust the field names and values based on your dataset

filtered_records = await client.get_records(
    dataset_id,
    where='local_authority="Surrey" OR local_authority="Kent"'
)

print(f"\nFiltered results: {filtered_records.total_count} total records")
print(f"Showing first {len(filtered_records.records)} records\n")

for i, record in enumerate(filtered_records.records, 1):
    print(f"Record {i} (ID: {record.id})")
    print(record.fields)
    print("-" * 40)


Filtered results: 23 total records
Showing first 10 records

Record 1 (ID: None)
{'smart': 5439, 'not_smart': 2549, 'percentage_smart': 68.08963445167751, 'geo_shape': {'type': 'Feature', 'geometry': {'coordinates': [[[[-0.509720585, 51.469175101], [-0.510296121, 51.467504502], [-0.50629852, 51.46771174], [-0.501459601, 51.467361346], [-0.497559413, 51.466181083], [-0.494557993, 51.463632862], [-0.489925221, 51.461896274], [-0.485340408, 51.461284996], [-0.476447848, 51.46113383], [-0.473969426, 51.458791527], [-0.471484989, 51.458228596], [-0.460464702, 51.457035051], [-0.45866924, 51.456304622], [-0.459314646, 51.452715326], [-0.461340498, 51.452391241], [-0.461502325, 51.448992476], [-0.457921231, 51.449222402], [-0.457464557, 51.44299912], [-0.455420558, 51.441952506], [-0.456488381, 51.438107108], [-0.446309632, 51.439996776], [-0.44773784, 51.435003387], [-0.439773431, 51.434619786], [-0.440004337, 51.430626653], [-0.432920264, 51.429049397], [-0.427852645, 51.429252224], [-0.42

### Selecting Specific Fields

Use `select` to return only the columns you need.
This makes responses smaller and easier to read, especially for wide datasets.

In [ ]:
# Select only specific fields to reduce response size
try:
    # Adjust field names based on your dataset
    records = await client.get_records(
        dataset_id,
        limit=5,
        # select="field1, field2",  # Uncomment and adjust
    )

    print(f"Retrieved {len(records.records)} records")

    # Display as a simple table
    for record in records.records:
        if record.fields:
            print(record.fields)

except Exception as e:
    print(f"Error: {e}")

### Pagination

Large datasets are usually returned in pages.
Use `limit` (page size) and `offset` (start position) to move through results page by page.

In [ ]:
# Paginate through records
page_size = 10
page_number = 0  # 0-indexed

try:
    records = await client.get_records(
        dataset_id,
        limit=page_size,
        offset=page_number * page_size,
    )

    total_pages = (records.total_count + page_size - 1) // page_size

    print(f"Page {page_number + 1} of {total_pages}")
    print(
        f"Showing records {page_number * page_size + 1} to {page_number * page_size + len(records.records)}"
    )
    print(f"Total records: {records.total_count}")

except Exception as e:
    print(f"Error: {e}")

## 5. Exporting Data to Different Formats

You can export data in formats that fit different tools and workflows:

- `json`: good for APIs and nested structures
- `csv`: simple tabular text, works in many tools
- `xlsx`: Excel workbook format
- `geojson`: geospatial JSON format
- `shapefile`: common GIS exchange format
- `kml`: map format used by tools like Google Earth

In [ ]:
from ukpyn import EXPORT_FORMATS

print("Available export formats:")
for fmt in EXPORT_FORMATS:
    print(f"  - {fmt}")

### Export to CSV

CSV is a good default when you want to inspect or share table-like data quickly.

In [ ]:
# Export dataset to CSV
try:
    from pathlib import Path

    csv_data = await client.export_data(
        dataset_id,
        format="csv",
        limit=100,  # Limit to 100 records for this example
    )

    save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
    if save_dir:
        output_file = Path(save_dir) / "export.csv"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "wb") as f:
            f.write(csv_data)
        print(f"Exported {len(csv_data)} bytes to {output_file}")
    else:
        print(
            f"Exported {len(csv_data)} bytes (file save skipped; set save_dir to enable writing)."
        )

    # Preview first few lines
    print("\nPreview (first 500 characters):")
    print(csv_data.decode("utf-8")[:500])

except Exception as e:
    print(f"Export error: {e}")

### Export to JSON

JSON is useful when you want structured data for scripts, apps, or APIs.

In [ ]:
import json

# Export dataset to JSON
try:
    json_data = await client.export_data(
        dataset_id,
        format="json",
        limit=10,
    )

    # Parse and pretty-print
    data = json.loads(json_data)
    print(json.dumps(data[:2], indent=2))  # Show first 2 records

except Exception as e:
    print(f"Export error: {e}")

### Export to Excel

Use Excel export when your audience prefers spreadsheet files.

In [ ]:
# Export dataset to Excel
try:
    from pathlib import Path

    xlsx_data = await client.export_data(
        dataset_id,
        format="xlsx",
        limit=100,
    )

    save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
    if save_dir:
        output_file = Path(save_dir) / "export.xlsx"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "wb") as f:
            f.write(xlsx_data)
        print(f"Exported {len(xlsx_data)} bytes to {output_file}")
        print("Open the file in Excel or use pandas to read it.")
    else:
        print(
            f"Exported {len(xlsx_data)} bytes (file save skipped; set save_dir to enable writing)."
        )

except Exception as e:
    print(f"Export error: {e}")

### Working with pandas (optional)

If you use pandas, you can load exports into a DataFrame for analysis and charting.

In [ ]:
# Optional: Use pandas for data analysis
# Make sure pandas is installed: pip install pandas

try:
    from io import BytesIO

    import pandas as pd

    # Export to CSV and load into pandas
    csv_data = await client.export_data(
        dataset_id,
        format="csv",
        limit=100,
    )

    # Create DataFrame
    df = pd.read_csv(BytesIO(csv_data), sep=";")

    print(f"DataFrame shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print("\nFirst 5 rows:")
    display(df.head())

except ImportError:
    print("pandas is not installed. Install it with: pip install pandas")
except Exception as e:
    print(f"Error: {e}")

## Cleaning Up

Close the client when you are finished to release network resources cleanly.

In [ ]:
# Close the client when done
await client.close()
print("Client closed.")

### Using Context Managers (Recommended)

`async with` is the safest pattern because the client is closed automatically, even if an error happens.

In [ ]:
# Recommended: Use async context manager for automatic cleanup
async with UKPNClient() as client:
    datasets = await client.list_datasets(limit=3)
    print(f"Found {datasets.total_count} datasets")

# Client is automatically closed when exiting the 'async with' block
print("Client automatically closed!")

## Error Handling

`ukpyn` raises specific exception types so you can show clearer messages and recover gracefully.

In [ ]:
from ukpyn import (
    AuthenticationError,
    NotFoundError,
    RateLimitError,
    ServerError,
    UKPNError,
    ValidationError,
)

async with UKPNClient() as client:
    try:
        # Try to access a non-existent dataset
        dataset = await client.get_dataset("this-dataset-does-not-exist")

    except NotFoundError as e:
        print(f"Dataset not found: {e}")

    except AuthenticationError as e:
        print(f"Authentication failed: {e}")
        print("Tip: Check that your UKPN_API_KEY is correct.")

    except RateLimitError as e:
        print(f"Rate limit exceeded: {e}")
        if e.retry_after:
            print(f"Try again in {e.retry_after} seconds.")

    except ValidationError as e:
        print(f"Invalid request: {e}")

    except ServerError as e:
        print(f"Server error: {e}")
        print("Tip: The UKPN API might be experiencing issues. Try again later.")

    except UKPNError as e:
        # Catch-all for other API errors
        print(f"API error: {e}")

## Summary

You now know how to:

- Install `ukpyn`
- Configure `UKPN_API_KEY`
- Create a `UKPNClient` and list datasets
- Search datasets and inspect their fields
- Fetch records with filtering, field selection, and pagination
- Export data in common formats
- Handle common API errors

## Next Steps

- Browse datasets on the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/)
- Learn basic [ODSQL filtering](https://help.opendatasoft.com/apis/ods-explore-v2/#section/Opendatasoft-Query-Language-(ODSQL))
- Continue with the next `ukpyn` tutorials for domain-specific examples

You are ready to start exploring real UKPN open data.